
# Algoritmo QR práctico
Recordamos la iteración QR de Hessenberg en la cual tenemos que $H = U^T_0AU_0$ donde $A,U\in \mathbb{R}^{n\times n}$ con $U$ ortogonal. Para cada una de las iteraciones lo que hacemos es tomar la factorización QR de $H = UR$, ($R\in \mathbb{R}^{n\times n}$ una matriz triangular superior )  y el siguiente paso $H=RU$. El objetivo es estudiar la convergencia de $H$ y como se puede mejorar haciendo uso de variaciones.

## Iteración QR con desplazamiento

Sea $\mu \in \mathbb{R}$ vamos a considerar la siguiente modificación para la iteración QR:
$$
\begin{align*}
&H = U^T_0AU_0\\
&\textbf{for } k = 1,2,\dots \\
&   \quad \quad \text{Determinar escalar } \mu \\
&    \quad \quad H - \mu I = UR \\
&   \quad \quad H = RU+\mu I \\
&\textbf{end}
\end{align*}
$$
$\mu$ es la variación. Podemos ver que cada una de las matrices generadas en este algoritmo es semejante a $A$ puesto que,
$$
H = RU +\mu I = U^TU(RU +\mu I) = U^TURU+U^TU\mu =U^T(UR+\mu I)U = U^THU
$$
Este razonamiento puede ser más claro si se hace por inducción, donde finalmente vamos a tener que 
$$
H = V^TAV
$$
donde $V$ es el producto de matrices $U$.

Si ordenamos los valores propios $\lambda_i$ de $A$ tenemos que
$$
|\lambda_1 - \mu|\geq |\lambda_2 - \mu|\geq \cdots \geq |\lambda_n - \mu|,
$$
con $\mu$ fijo en cada iteración, por lo visto antes en la sección de iteraciones de potencias tenemos que la $p-$ésima entrada de la sibdiagonal de $H$ converge a cero con una tasa de 
$$
\left|\frac{\lambda_{p+1}-\mu}{\lambda_p-\mu}\right|^k
$$

Ahora, es claro que si $\lambda_p = \lambda_{p+1}$ no hay convergencia, y para el caso en que $\mu \to \lambda_n$ la entrada $(n,n-1)$ se acera rápido.

Veamos ahora como resulta el programa al hacer el algoritmo anterior.

In [1]:
using LinearAlgebra

#Para este notebook vamos a necesitar de house, Givens y HessenbergForm
#house
function house(x)
    y = copy(x)
    n = size(x,1) #(m,n)
    s = x[2:n]'x[2:n]
    v = [1; x[2:n]]
    
    if s == 0
        β = 0
        
    else
        mu = sqrt(x[1]^2 + s)
        if x[1] <= 0
            v[1] = x[1] - mu
        else
            v[1] = -s /(x[1]+mu)
        end
        β = 2((v[1])^2)/(s + (v[1])^2)
        v=v/v[1]
    end
    return v, β
end
#Givens
function Givens(a,b)
    if b==0
        c = 1
        s = 0
    else
        if abs(b)>abs(a)
            τ=-a/b
            s=-1/sqrt(1+τ^2)
            c=s*τ
        else
            τ=-b/a
            c=1/sqrt(1+τ^2)
            s=c*τ
        end
    end
    return c,s
end

function HessenbergForm(A)
    n = size(A)[1]
    H = copy(A)
    Q = Matrix(1.0*I, n, n)
    for k = 1:n-2
        v, β = house(H[k+1:n, k])
        H[k+1:n, k:n] = (I - β*v*v')*H[k+1:n, k:n]
        H[1:n, k+1:n] = H[1:n, k+1:n]*(I - β*v*v')
        
        #Q es necesaria para la verificar que la función devuelve los resultados correctos
        Q[1:n, k+1:n] = Q[1:n, k+1:n]*(I - β*v*v') 
    end
    return H, Q
end

HessenbergForm (generic function with 1 method)

In [2]:
# The Shifted  QR Iteration 1
function HessenbergQR1(H, μ)
    n = size(H)[1]
    K = μ*I
    H2 = copy(H) - K
    
    C, S = zeros(n-1), zeros(n-1)
    
    #Factorización QR de H
    for k = 1:n-1
        C[k], S[k] = Givens(H2[k,k], H2[k+1, k])
        H2[k:k+1,k:n] = [C[k] -S[k]; S[k] C[k]]*H2[k:k+1,k:n]
        H2[k+1,k] = 0
        #display([C[k] S[k]])
    end
    
    #Matriz RQ
    for k = 1:n-1
        H2[1:k+1, k:k+1] = H2[1:k+1, k:k+1]*[C[k] S[k]; -S[k] C[k]] #+ K[1:k+1, k:k+1]
        #display(K[1:k+1, k:k+1])
    end
    
    return H2+K #RQ Hessenberg superior
end

HessenbergQR1 (generic function with 1 method)

In [3]:
I+rand(2,2)

2×2 Matrix{Float64}:
 1.58466   0.680591
 0.737754  1.66685

Hagamos une prueba de como funciona el algoritmo para una matriz aleatoria.

In [4]:
A=5*rand(4,4)-2.0*I
H,Q =HessenbergForm(A)
H
H2 = HessenbergQR1(H,2.0)


4×4 Matrix{Float64}:
  2.09872      3.67974       3.2853    -1.06625
  6.84993      2.09283       3.26718    0.0256178
 -2.22045e-16  2.73603      -0.688322  -3.2364
 -6.66134e-16  1.11022e-16  -0.180616   0.39047

Podemos ver que claramente converge a una matriz de Hessenberg nuevamente. Ahora veamos el siguiente teorema, que nos da información en caso de que esta variación $\mu$ sea un valor propio de la matriz de Hessenberg $H$.

$\textbf{Teorema:}$ Sea $\mu$ un valor propio de una matriz de Hessenberg de tamaño $n\times n$ (no reducida). Si $\overline{H} = RU + \mu I$, donde $H - \mu I = UR$ es la factorización QR de $H-\mu I$, entonces $\overline{h}_{n,n-1}=0$ y $\overline{h}_{n,n}=\mu$.

$\textbf{Demostración:}$ Como $H$ es no reducida, vamos a tener en particular que las primeras $n-1$ columnas son independientes, de esta manera, vamos a tener que si consideramos $H-\mu I $ las primeras $n-1$ columnas también son independientes. Por lo tanto si $UR = H-\mu I$ es la factorización $QR$ vamos a tener que para cada $i=1, \dots, n-1$ $r_{ii}\neq 0$. Pero si $H-\mu I$ es singular entonces se tiene que $r_{nn} = 0$, y como $\overline{H} = RU + \mu I$ entonces, vamos a tener que $fl_n(\overline{H}) = [0,\dots,0,\mu]$.

Veamos esto ahora como un ejemplo, considerando una matriz y encontrando un valor propio de ella.

In [5]:
# Ejemplo 7.5.1 tomado del libro guía.
H = [9.0 -1 -2; 2 6 -2; 0 1 5]
eigvals(H)

3-element Vector{Float64}:
 6.000000000000002
 6.99999990953704
 7.000000090462961

In [6]:
H2 = HessenbergQR1(H,6.0)

3×3 Matrix{Float64}:
 8.53846   -3.73132   1.00901
 0.634324   5.46154  -1.38675
 0.0        0.0       6.0

In [7]:
#Hagamos otro ejemplo con una matriz aleatoria
B=5*rand(4,4)-2.0*I
H,Q = HessenbergForm(B)
H
eigvals(H)


4-element Vector{ComplexF64}:
 -2.3440955085428428 - 1.0425620035979735im
 -2.3440955085428428 + 1.0425620035979735im
 0.10976477161976443 + 0.0im
   8.332888127279661 + 0.0im

In [8]:
H2 = HessenbergQR1(H,7.00275)

4×4 Matrix{Float64}:
 -2.68281       0.649689      2.74082   2.25072
 -1.18005       0.173607     -3.41144   2.99959
 -2.22045e-16  -4.55273       6.66205  -1.29157
 -2.22045e-16  -2.22045e-16   1.17448  -0.398384

Hasta el momento lo que hemos hecho es dejar fijo a $\mu$, que tal si ahora variamos $\mu$ en cada una de las iteraciones. Al realizar esto, lo que vamos a obtener es una incoporación de nueva información sobre $\lambda(A)$. Una posible consideración es tomar $h_{nn}$ como la mejor aproximación a un valor propio en la diagonal. Si realizamos este cambio, en cada una de las iteraciones obtenemos "The Single-Shift QR iterarion"

$$
\begin{align*}
&H = U^T_0AU_0\\
&\textbf{for } k = 1,2,\dots \\
& \quad \quad\mu = H(n,n) \\
& \quad \quad H - \mu I = UR \\
& \quad \quad H = RU+\mu I \\
&\textbf{end}
\end{align*}
$$
Veamos esto en Julia:

In [9]:
# The Single-Shift QR Iteration 2
function HessenbergSSQR(H)
    n = size(H)[1]
    H2 = copy(H)
    C, S = zeros(n-1), zeros(n-1)
    μ = H2[n,n]
    K = μ*I
    H2 = H2 - K
    #Factorización QR de H
    for k = 1:n-1
        #println("primer for ", μ)
        C[k], S[k] = Givens(H2[k,k], H2[k+1, k])
        H2[k:k+1,k:n] = [C[k] -S[k]; S[k] C[k]]*H2[k:k+1,k:n]
        H2[k+1,k] = 0
        #display(H2)
    end
    #display(aux)
    
    
    #Matriz RQ
    for k = 1:n-1
        H2[1:k+1, k:k+1] = H2[1:k+1, k:k+1]*[C[k] S[k]; -S[k] C[k]]
        #display(K[1:k+1, k:k+1])
    end
    
    return H2+K #RQ Hessenberg superior
end

HessenbergSSQR (generic function with 1 method)

Veamos ahora un ejemplo en particular para este nuevo algoritmo:

In [10]:
H = [1.0 2 3; 4 5 6; 0 0.001 7]

3×3 Matrix{Float64}:
 1.0  2.0    3.0
 4.0  5.0    6.0
 0.0  0.001  7.0

In [11]:
H2= HessenbergSSQR(H)
display(H2)


3×3 Matrix{Float64}:
 -0.538462  -1.6908      -0.8351
  0.307693   6.52646      6.65555
  0.0       -2.16332e-5   7.012

Con el algoritmo anterior podemos presentar dificultades en el caso que la matriz 
$$
G = \begin{bmatrix}
h_{n-1n-1} & h_{n-1n} \\
h_{nn-1} & h_{nn}
\end{bmatrix}
$$
tenga sus dos valores propios en el campo de los complejos, por lo que $h_{nn}$ tiende a ser una mala aproximación de un valor propio. 

Para solucionar este problema lo que hacemos es desarrollar la estrategia de variación doble, tomando como parámetros de variación a estos dos valores complejos $\lambda_1, \lambda_2$, esto es:
$$
\begin{align*}
H - \lambda_1I &= U_1R_1, \\
H_1 &= R_1U_1 + \lambda_1I, \\
H_1 - \lambda_2I &= U_2R_2, \\
H_2 &= R_2U_2 + \lambda_2I.
\end{align*}
$$
Note que 
$$
(U_1U_2)(R_2R_1) = (H-\lambda_1I)(H-\lambda_2I) = M,
$$
puesto que,
$$
\begin{align*}
(U_1U_2)(R_2R_1) = U_1(U_2R_2)R_1 &= U_1(H_1 - \lambda_2I)R_1, \\
&= U_1(R_1U_1 + \lambda_1I - \lambda_2I)R_1, \\
&= U_1R_1(U_1R_1 + \lambda_1I - \lambda_2I), \\
& = U_1R_1(H - \lambda_1I + \lambda_1I - \lambda_2I),\\
& = U_1R_1(H - \lambda_2I),\\
& = (H - \lambda_1I)(H - \lambda_2I) = M.
\end{align*}
$$
También podemos ver que $M$ es una matriz real incluso si los valores propios de $G$ son complejos ya que
$$
\begin{align*}
M &= (H - \lambda_1I)(H - \lambda_2I), \\
  &= H^2 -(\lambda_2 + \lambda_1)H + \lambda_1\lambda_2I.
\end{align*}
$$
Note que como $G$ tiene dos valores propios diferentes, entonces $G$ es diagonalizable y por lo tanto tenemos que
$$
\lambda_2 + \lambda_1 = h_{n-1n-1} + h_{nn} = \text{tr}(G)\in \mathbb{R},
$$
y por el mismo argumento tenemos que
$$
\lambda_1\lambda_2 = h_{n-1n-1}h_{nn} = \det(G)\in \mathbb{R}.
$$
    Podemos concluir que $M$ es la factorización QR de una matriz real, podemos escoger entonces $Z=U_1U_2$ es una matriz ortogonal real y entonces tenemos que $H_2$ es real. Para justificar esa afirmación recordamos la iteración QR en donde
$$
H_2 = U^H_2H_1U_2 =  U^H_2(U^H_1HU_1)U_2 = (U_1U_2)^HHU_1U_2 = Z^THZ.
$$

Desafortunadamente los errores de redondeo casi siempre evita que el resultado sea real, podemos garantizar que $H_2$ tiene entradas reales si:
1. Formamos la matriz $M$ explicitamente.
2. Calculamos la factorización QR $M=ZR$.
3. $H_2 = Z^THZ$
Sin embargo no es práctico debido a que el punto 1. requiere del orden de $O(n^3)$ iteraciones para ser caclulado.


## Estrategia de variación doble implícita
Se puede realizar la transición de $H$ a $H_2$ en $O(n^2)$ recurriendo al teorema de la función implícita, recordemos que nos dice este teorema:

$\textbf{Teorema de la Q implícita:}$ Suponga que $Q = [q_1, \dots, q_n]$ y que $V = [v_1, \dots, v_n]$ son ortogonales tales que $Q^TAQ = H$ y $V^TAV = G$ matrices de Hessenberg, donde $A\in \mathbb{R}^{n\times n}$. Sea $k$ el entero positivo más pequeño para el que se cumple que $h_{k+1,k}=0$ con la convención de que $k=n$ si $H$ es no reducida. Si $q_1 = v_1$ entonces $q_i = \pm v_i$ y $|h_{i,i-1}|=|g_{i,i-1}|$ para $i = 2, \dots k$. Más aún, si $k<n$, $g_{k+1,k}=0$.

Para realizar la transición de $H$ a $H_2$ podemos proceder haciendo lo siguiente:

1. Calculamos $Me_1$ (la primera columna de M).
2. Determinamos la matriz de Householder $P_0$ tal que $P_0(Me_1)$ es un múltiplo de $e_1$.
3.Calculamos las matrices de Householder $P_1, P_2, \dots P_{n-2}$ tal que si $Z_1 = P_0P_1\cdots P_{n-2}$ entonces $Z_1HZ_1$ es una matriz de Hessenberg y la primera columan de $Z$ y $Z_1$ son iguales.

Bajo estas hipótesis, el teorema de la Q implícita nos permite concluir que si $Z^T_1HZ_1$ y $Z^T_1HZ_1$ son matrices de Hessenberg no reducidas entonces esencialmente son las mismas matrices.

Estudiemos un poco los detalles, $P_0$ se puede determinar sin tanto esfuerzo, considerando que 
$$
Me_1 = [x,y,z, 0 , \dots , 0]^T,
$$
donde
$$
\begin{align*}
x & = h^2_{11}+h_{12}h_{21} - (\lambda_1 + \lambda_2)h_{11} + \lambda_1\lambda_2, \\
y & = h_{21}(h_{11}+h_{22}-(\lambda_1 + \lambda_2)),\\
z & = h_{21}h_{32}.
\end{align*}
$$
Para ver esto, basta recordar que $M = H^2 -(\lambda_1+\lambda_2)H + \lambda_1\lambda_2I$, y haciendo los cálculos respectivos.

Puesto que una transformación semejante con $P_0$ solo cambia filas y las columnas 1,2 y 3, tenemos que 

$$
F = P_0HP_0 =\begin{bmatrix}
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
      0&0 & 0 & \times & \times & \times \\
0 & 0 & 0 & 0 & \times & \times \\
\end{bmatrix}.
$$
El objetivo de las matrices $P_i$ con $i=1,\dots, n-2$ es volver a $F$ una matriz de Hessenberg de nuevo. Para hacernos una idea de como funciona estas operaciones lo podemos ilustrar de la siguiente forma:

$$
\begin{align*}
&\begin{bmatrix}
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
      0&0 & 0 & \times & \times & \times \\
0 & 0 & 0 & 0 & \times & \times \\
\end{bmatrix} \xrightarrow{P_1} \begin{bmatrix}
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
0 & \times & \times & \times & \times & \times \\
0 & \times & \times & \times & \times & \times \\
 0&\times & \times & \times & \times & \times \\
0 & 0 & 0 & 0 & \times & \times \\
\end{bmatrix} \xrightarrow{P_2}  \begin{bmatrix}
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
0 & \times & \times & \times & \times & \times \\
0 & 0 & \times & \times & \times & \times \\
 0&0 & \times & \times & \times & \times \\
0 & 0 & \times & \times & \times & \times \\
\end{bmatrix} \xrightarrow{P_3} \\
&  \begin{bmatrix}
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
0 & \times & \times & \times & \times & \times \\
0 & 0 & \times & \times & \times & \times \\
 0&0 & 0 & \times & \times & \times \\
0 & 0 & 0 & \times & \times & \times \\
\end{bmatrix} \xrightarrow{P_4} \begin{bmatrix}
\times & \times & \times & \times & \times & \times \\
\times & \times & \times & \times & \times & \times \\
0 & \times & \times & \times & \times & \times \\
0 & 0 & \times & \times & \times & \times \\
 0&0 & 0 & \times & \times & \times \\
0 & 0 & 0 & 0 & \times & \times \\
\end{bmatrix}.
\end{align*}
$$
En general, $P_k$ tiene la forma $P_k=\text{diag}(I_k,D_k,I_{n-k-3})$ donde $D_k$ es una matriz de Householder $3\times 3$ por ejemplo:
$$
P_1 = \begin{bmatrix}
1 & 0 & 0 & 0 & 0 & 0 \\
0 & \times & \times & \times & 0 & 0\\
0 & \times & \times & \times & 0& 0 \\
0 & \times & \times & \times & 0& 0 \\
 0&0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0& 1\\ 
\end{bmatrix}
$$
Para $P_{n-2}$ tenemos una excepción, quedando asi $P_{n-2}=\text{diag}(I_{n-2},D_{n-2})$

El hecho de aplicar el teorema de la $Q$ implícita se sigue del hecho de que $P_ke_1 = e_1$ para $k = 1, \dots, n-2$ y que $P_0$ y $Z$ la primera columna es igual, por lo tanto $Z_1e_1 = Ze_1$ y podemos afirmar que $Z_1$ es esencialmente $Z$ siempre que se tenga que $Z^THZ,Z^T_1HZ_1$ son matrices de Hessenberg no reducidas.

La determinación implícita de $H_2$ de $H$ fue hecha por Francis, y por eso lo hecho anteriormente se le conoce como el paso QR de Francis.

Este algoritmo se presenta a continuación.


In [30]:
#Algoritmo Francis QR step. 3
function FQR(H)
    n = size(H)[1]
    m = n-1
    #Calculo de la primera columna de M = (H-a1I)(H-a2I)
    s = H[m,m] + H[n,n]
    t = H[m,m]*H[n,n] - H[m,n]*H[n,m]
    x = H[1,1]*H[1,1] + H[1,2]*H[2,1]-s*H[1,1]+t
    y = H[2,1]*(H[1,1]+H[2,2]-s)
    z = H[2,1]*H[3,2]
    for k = 0:n-3
        v, β = house([x, y, z])
        q = max(1,k)
        H[k+1:k+3, q:n] = (I-β*v*v')*H[k+1:k+3, q:n]
        r = min(k+4,n)
        H[1:r, k+1:k+3] = H[1:r, k+1:k+3]*(I-β*v*v')
        x = H[k+2,k+1]
        y = H[k+3,k+1]
        if k < n-3
            z = H[k+4,k+1]
        end
    end
    v, β = house([x, y])
    H[n-1:n,n-2:n] = (I-β*v*v')*H[n-1:n,n-2:n]
    H[1:n,n-1:n] = H[1:n,n-1:n]*(I-β*v*v')
    return H
end 

FQR (generic function with 1 method)

In [13]:
C = 4*rand(5,5)+2*I

5×5 Matrix{Float64}:
 3.35945  3.61487   1.44907  3.56865  1.94725
 3.65201  2.6777    1.12     1.08502  2.08718
 2.9483   1.13517   2.50936  2.88195  3.87045
 1.20429  0.593915  3.32768  3.28839  1.63753
 1.80649  3.30448   1.67906  1.0882   4.27182

In [14]:
H1, Q =HessenbergForm(C)
H1

5×5 Matrix{Float64}:
  3.35945       4.89021       1.3915       -0.573263  -2.34843
  5.1714        7.51614       3.70093      -0.899556  -0.0517066
  0.0           3.55904       2.22527       0.324292   0.221855
 -2.22045e-16  -2.10335e-17   2.04241       2.0475    -0.385196
  4.44089e-16   0.0          -4.71845e-16  -2.17201    0.958351

In [15]:
FH = FQR(H1)

5×5 Matrix{Float64}:
 12.046        -0.370369     0.764797     -1.05111    -0.90527
  1.14815      -0.589579     0.417253      0.961139    1.23152
  5.55112e-17   1.46254      1.97454      -0.0335565   1.8256
 -1.66533e-16   0.0          1.71322       0.50302    -1.08066
  4.44089e-16  -1.94289e-16  3.33067e-16  -1.47135     2.17278

In [16]:
function HessenbergQR(H)
    n = size(H)[1]
    H2 = copy(H)
    C, S = zeros(n-1), zeros(n-1)
    
    #Factorización QR de H
    for k = 1:n-1
        C[k], S[k] = Givens(H2[k,k], H2[k+1, k])
        H2[k:k+1,k:n] = [C[k] -S[k]; S[k] C[k]]*H2[k:k+1,k:n]
        H2[k+1,k] = 0
        #display([C[k] S[k]])
    end
    
    #Matriz RQ
    for k = 1:n-1
        H2[1:k+1, k:k+1] = H2[1:k+1, k:k+1]*[C[k] S[k]; -S[k] C[k]]
    end
    
    return H2 #RQ Hessenberg superior
end
#Real-Shur sin modificar la iteracion de Hessenberg
function RealSchur(A, iteraciones = 10000)
    H0 = A
    H1, Q = HessenbergForm(A)
    for k = 1:iteraciones
        H0 = H1
        H1 = HessenbergQR(H1)
    end
    return H1
    
end
#Real-Shur modificando la iteración de Hessenberg con The Shifted  QR Iteration
function RealSchur1(A,μ, iteraciones = 10000)
    H0 = A
    H1, Q = HessenbergForm(A)
    for k = 1:iteraciones
        H0 = H1
        H1 = HessenbergQR1(H1,μ)
    end
    return H1
end
#Real-Shur modificando la iteración de Hessenberg con The Single-Shift QR Iteration
function RealSchur2(A, iteraciones = 1000)
    H0 = A
    H1, Q = HessenbergForm(A)
    for k = 1:iteraciones
        H0 = H1
        H1 = HessenbergSSQR(H1)
    end
    return H1
end
#Real-Shur haciendo la iteracion de QR de Francis
function RealSchur3(A, iteraciones = 5000)
    H0 = A
    H1, Q = HessenbergForm(A)
    for k = 1:iteraciones
        H0 = H1
        H1 = FQR(H1)
    end
    return H1
end
function autovComplejos(H)
    D = diag(H,-1)
    for k = 1:size(D)[1]
        if abs(D[k])>10^-12
            display(eigvals(H[k:k+1,k:k+1]))
        end
    end
end

autovComplejos (generic function with 1 method)

In [17]:
#Pruebas
#A = 3*rand(5,5)+2*I
A = [9.0 -1 -2; 2 6 -2; 0 1 5]
println("Valores propios de A:")
display(eigvals(A))
println("Resultado de la iteración es:")
display(RealSchur(A))
println("Resultado de la iteración Shifted es:")
display(RealSchur1(A,1))
println("Resultado de la iteración Single-Shifted es:")
display(RealSchur2(A))
println("Resultado de la iteración QR de Francis es:")
display(RealSchur3(A))

Valores propios de A:


3-element Vector{Float64}:
 6.000000000000002
 6.99999990953704
 7.000000090462961

Resultado de la iteración es:


3×3 Matrix{Float64}:
 7.0007     -4.36564    0.404493
 1.1231e-7   6.9993    -1.6666
 0.0         9.0e-323   6.0

Resultado de la iteración Shifted es:


3×3 Matrix{Float64}:
 7.0006      -4.36564    0.404455
 8.25061e-8   6.9994    -1.66661
 0.0          7.4e-323   6.0

Resultado de la iteración Single-Shifted es:


3×3 Matrix{Float64}:
 7.001       -4.36564   0.404608
 2.29772e-7   6.999    -1.66657
 0.0          0.0       6.0

Resultado de la iteración QR de Francis es:


3×3 Matrix{Float64}:
  7.0           -4.36564       0.404226
  1.01804e-180   7.0          -1.66667
 -2.96854e-170   8.0783e-158   6.0

In [35]:
using BenchmarkTools

Ahora hagamos pruebas de eficiencia haciendo cada uno de los cambios mostrados anteriormente:
$\textit{Shifted, Single-Shifted y la iteración QR de Francis}$.

In [36]:
n = 50
D = 6*rand(n,n)+2*I

50×50 Matrix{Float64}:
 5.74751    4.32937   4.14345   3.35771   …  1.54352   4.90918   1.69188
 1.31073    6.07638   4.43489   0.296723     1.57838   3.44437   1.75567
 5.75712    2.1442    4.48793   5.11605      3.72451   1.25975   5.48005
 3.89524    5.47145   3.61715   6.9942       3.04174   3.90274   3.26644
 5.66905    1.82347   5.59119   2.14157      1.57305   4.24318   1.88831
 0.176445   0.174959  1.88852   5.50352   …  4.81707   5.85711   0.0326821
 0.985017   5.13566   4.96122   2.18806      5.73748   2.11362   5.08418
 1.71606    3.045     2.4828    3.71019      1.27968   5.14641   4.94169
 4.02182    4.70407   5.07202   3.71365      5.88469   1.97906   2.56667
 0.0193767  1.51748   3.80821   5.63858      2.31831   3.81676   1.14714
 4.64455    4.44317   1.42106   5.10576   …  4.6799    3.76833   0.0426059
 2.34616    2.0881    1.82222   4.08718      4.06246   3.12556   4.51928
 5.29187    4.37999   5.58294   0.198645     4.04302   1.27173   1.39004
 ⋮                      

In [56]:
l = eigvals(D)

50-element Vector{ComplexF64}:
    -8.4118064365097 + 0.0im
   -7.81401796629787 - 0.47576851017383603im
   -7.81401796629787 + 0.47576851017383603im
 -6.9851225711709155 - 7.958961643891215im
 -6.9851225711709155 + 7.958961643891215im
 -6.7206615620347705 - 4.997959336797278im
 -6.7206615620347705 + 4.997959336797278im
  -5.861366137384971 - 4.026312891339715im
  -5.861366137384971 + 4.026312891339715im
 -3.5570858135091914 - 2.336428927497644im
 -3.5570858135091914 + 2.336428927497644im
 -3.0312250570101402 - 8.340750672662347im
 -3.0312250570101402 + 8.340750672662347im
                     ⋮
   8.895657914460743 + 5.42014316635007im
    9.61736734437489 + 0.0im
  10.140835465997174 - 4.294363867609314im
  10.140835465997174 + 4.294363867609314im
  10.947855962107212 + 0.0im
  10.955083162709592 - 6.448857102764402im
  10.955083162709592 + 6.448857102764402im
  12.538056459215996 - 3.2031301830547805im
  12.538056459215996 + 3.2031301830547805im
   13.38661737875216 + 0.0im
  15.572

In [55]:
y = 5 in l

false

In [47]:
@benchmark RealSchur(D)


BenchmarkTools.Trial: 9 samples with 1 evaluation.
 Range (min … max):  594.186 ms … 596.419 ms  ┊ GC (min … max): 6.22% … 6.49%
 Time  (median):     595.099 ms               ┊ GC (median):    6.22%
 Time  (mean ± σ):   595.092 ms ± 701.089 μs  ┊ GC (mean ± σ):  6.27% ± 0.11%

  █ █         █         █ ██      █        █                  █  
  █▁█▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁█▁██▁▁▁▁▁▁█▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  594 ms           Histogram: frequency by time          596 ms <

 Memory estimate: 1.29 GiB, allocs estimate: 7881130.

In [51]:
@benchmark RealSchur1(D,5)

BenchmarkTools.Trial: 8 samples with 1 evaluation.
 Range (min … max):  626.163 ms … 654.230 ms  ┊ GC (min … max): 6.80% … 7.47%
 Time  (median):     653.253 ms               ┊ GC (median):    7.43%
 Time  (mean ± σ):   647.615 ms ±  10.263 ms  ┊ GC (mean ± σ):  7.33% ± 0.27%

                                                              █  
  ▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▁▁▁▁▁▁▁▁▁▁▇▇█ ▁
  626 ms           Histogram: frequency by time          654 ms <

 Memory estimate: 1.67 GiB, allocs estimate: 7921130.

In [49]:
@benchmark RealSchur2(D)

BenchmarkTools.Trial: 78 samples with 1 evaluation.
 Range (min … max):  63.152 ms …  68.058 ms  ┊ GC (min … max): 6.16% … 8.17%
 Time  (median):     63.755 ms               ┊ GC (median):    6.52%
 Time  (mean ± σ):   64.114 ms ± 795.220 μs  ┊ GC (mean ± σ):  7.10% ± 0.91%

      ▃   ▃ ▆█ ▆                                 ▃              
  ▄▁▇▄█▇▁▄█▇██▇█▇▇▄▄▄▄▄▇▁▄▄▄▁▁▁▁▁▁▄▁▄▄▄▄▄▄▇▄▇▄▇▄▄█▁▄▄▁▁▁▄▇▁▁▄▄ ▁
  63.2 ms         Histogram: frequency by time         65.4 ms <

 Memory estimate: 174.70 MiB, allocs estimate: 793130.

In [50]:
@benchmark RealSchur3(D)

BenchmarkTools.Trial: 12 samples with 1 evaluation.
 Range (min … max):  434.227 ms … 436.119 ms  ┊ GC (min … max): 4.89% … 5.16%
 Time  (median):     435.422 ms               ┊ GC (median):    5.15%
 Time  (mean ± σ):   435.399 ms ± 501.873 μs  ┊ GC (mean ± σ):  5.09% ± 0.12%

  ▁                ▁                █ █   ▁   ▁▁    ▁▁        ▁  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁█▁▁▁█▁▁▁██▁▁▁▁██▁▁▁▁▁▁▁▁█ ▁
  434 ms           Histogram: frequency by time          436 ms <

 Memory estimate: 946.94 MiB, allocs estimate: 3995642.

Nos podemos dar cuenta que en terminos de tiempo, $\textit{Single-Shifted y la iteración QR de Francis}$ son mejores que la iteración inicial, pero ¿qué paso con shifted? en comparación de las otras tres iteraciones el tiempo que se requiere es mucho mayor, notemos que el $\mu$ escogido no es un valor propio de la matriz, miremos que pasa ahora si tomamos $\mu$ como un valor propio de la matriz $D$

In [57]:
@benchmark RealSchur1(D,9.61736734437489)

BenchmarkTools.Trial: 8 samples with 1 evaluation.
 Range (min … max):  630.420 ms … 632.352 ms  ┊ GC (min … max): 6.75% … 7.04%
 Time  (median):     631.206 ms               ┊ GC (median):    6.91%
 Time  (mean ± σ):   631.328 ms ± 747.155 μs  ┊ GC (mean ± σ):  6.92% ± 0.11%

  █ █          █      █        █                 █       █    █  
  █▁█▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁█▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁█▁▁▁▁█ ▁
  630 ms           Histogram: frequency by time          632 ms <

 Memory estimate: 1.67 GiB, allocs estimate: 7921130.

Podemos ver que baja el tiempo con respecto a la iteración cuando $\mu$ no es un valor propio, sin embargo, con respecto a las demás, el tiempo de la iteración sigue siendo mayor. 